# 1. API 설정

In [26]:
import os
from dotenv import load_dotenv

load_dotenv()

True

# 2. vector store 설정

In [27]:
from openai import OpenAI

client = OpenAI()

vector_store = client.vector_stores.create(
    name = "korean-history" # korean history를 바탕
)

print(f"Vector store 생성 완료")
print(f"ID {vector_store.id}")
print(f"Name {vector_store.name}")
print(f"Status {vector_store.status}")

Vector store 생성 완료
ID vs_6a1ee6d2219c8191b699a93015d32a65
Name korean-history
Status completed


# 3. PDF 업로드

In [28]:
from pypdf import PdfReader, PdfWriter

pdf_path = "수능특강_한국사.pdf"
output_path = "수능특강_한국사_6_7페이지.pdf"

reader = PdfReader(pdf_path)
writer = PdfWriter()

# PDF 페이지는 0부터 시작
# 6페이지 -> index 5
# 7페이지 -> index 6
for page_num in [5, 6]:
    writer.add_page(reader.pages[page_num])

with open(output_path, "wb") as f:
    writer.write(f)

print(f"저장 완료: {output_path}")

저장 완료: 수능특강_한국사_6_7페이지.pdf


In [29]:
with open(output_path, "rb") as f:
    file_upload = client.files.create(
        file=f,
        purpose="assistants" # "assistants": 지식 기반, "fine-tuning": 미세조정
    )

print("파일 업로드 완료!")
print(f"ID {file_upload.id}")
print(f"Filename {file_upload.filename}")
print(f"Size {file_upload.bytes} bytes")

파일 업로드 완료!
ID file-2aKWnxFt8gTe8US41rvK8o
Filename 수능특강_한국사_6_7페이지.pdf
Size 241685 bytes


In [30]:
# vector store에 파일 인덱싱
indexed_file = client.vector_stores.files.create_and_poll(
    vector_store_id = vector_store.id,
    file_id = file_upload.id
)

print(f"파일 인덱싱 완료")
print(f"ID {indexed_file.id}")
print(f"Status {indexed_file.status}")

파일 인덱싱 완료
ID file-2aKWnxFt8gTe8US41rvK8o
Status completed


In [31]:
# vector store 상태 확인
vs_status = client.vector_stores.retrieve(vector_store.id)

print(f"Vector store 상태")
print(f"총 파일 수: {vs_status.file_counts.total}")
print(f"완료된 파일: {vs_status.file_counts.completed}")
print(f"처리 중인 파일: {vs_status.file_counts.in_progress}")

Vector store 상태
총 파일 수: 1
완료된 파일: 1
처리 중인 파일: 0


# 4. 에이전트 만들기

In [32]:
from agents import Agent, FileSearchTool, Runner, trace

rag_agent = Agent(
    name = "RAG Expert", # RAG 전문가
    instructions = """You are a helpful assistant that answers questions based on the documents in the vector store.""",
    tools = [
        FileSearchTool(
            vector_store_ids = [vector_store.id],
            max_num_results = 5, # 상위 5개
            include_search_results = True
        )
    ]
)

print(f"Agent 생성 완료")
print(f"Name: {rag_agent.name}")
print(f"Tools: {[tool.name for tool in rag_agent.tools]}")

Agent 생성 완료
Name: RAG Expert
Tools: ['file_search']


In [33]:
async def ask_rag_agent(question: str): # 비동기: 여러 질문을 한 번에 처리할 수 있도록 함
    """RAG 에이전트에게 질문하기"""
    with trace("RAG Query"):
        result = await Runner.run(rag_agent, question) # await rag_agent.run(question)
        return result

# 첫 번째 질문하기
question1 = "청동기 시대 사람들의 대표적인 무덤이 뭐야?"

print(f"질문: {question1}")
print("=" * 50)

result1 = await ask_rag_agent(question1) 
# await: 할 때까지 기다리겠다는 의미, if not -> 나중에 처리할 작업으로 넘김
print(f"답변: {result1}")

질문: 청동기 시대 사람들의 대표적인 무덤이 뭐야?
답변: RunResult:
- Last agent: Agent(name="RAG Expert", ...)
- Final output (str):
    청동기 시대의 대표적인 무덤은 고인돌입니다.
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


In [35]:
print(result1.final_output)

청동기 시대의 대표적인 무덤은 고인돌입니다.


In [34]:
# for attr in dir(result1):
#     if not attr.startswith("_"):
#         print(attr)

# 5. 멀티 에이전트 만들기

In [41]:
from agents import Agent, FileSearchTool, Runner, trace

# 1. rag_specialist
rag_specialist = Agent(
    name = "rag_specialist",
    handoff_description="Answers questions using the Korean History vector store.",
    instructions = """You are a helpful assistant that answers questions based on the documents in the vector store.""",
    tools = [
        FileSearchTool(
            vector_store_ids = [vector_store.id],
            max_num_results = 3,
            include_search_results = True
        )
    ]
)

# 2. 일반
general_assistant = Agent(
    name="general_assistant",
    handoff_description="Handles casual conversations and non-Korean-history questions.",
    instructions="""
You are a friendly general assistant.

Your responsibilities:
- Handle greetings.
- Handle casual conversation.
- Handle questions unrelated to the Korean History knowledge base.
- Handle general world knowledge questions.

Do NOT answer questions about Korean history topics that may exist in the vector store.

Answer in Korean.
""", 
)

# 3. 요약
summarizer = Agent(
    name="summarizer",
    handoff_description="Creates summaries from the Korean History knowledge base.",
    instructions="""
You are a Korean History Summarization Specialist.

Your responsibilities:
- Search the vector store before creating summaries.
- Summarize retrieved information into clear and structured explanations.
- Focus on major concepts, chronology, causes, effects, and significance.

Use this agent when:
- The user asks for a summary.
- The user asks for an overview.
- The user asks for key points.
- The user asks to organize information.

Response guidelines:
- Use bullet points when appropriate.
- Keep summaries easy to understand.
- Answer in Korean.
""",

    tools = [
        FileSearchTool(
            vector_store_ids = [vector_store.id],
            max_num_results = 10, # 요약을 위해 더 많은 결과
            include_search_results = True
        )
    ]
)

In [42]:
triage_agent = Agent(
    name="triage_agent",
    instructions="""You are a triage agent that routes user questions to the appropriate specialist.

Routing rules:
1. ** RAG Specialist **: Questions about the Korean history(선사시대, 구석기, 신석기, 청동기, etc.)
2. ** Summarizer **: Requests for summa|ries, overviews, or key takeaways
3. ** General Assistant **: Greetings, general questions, or anything not related to the document

Always analyze the user's intent carefully before routing.
Do NOT answer questions directly - always hand off to the appropriate specialist.
""",
    handoffs=[rag_specialist, summarizer, general_assistant]
)

print(f"Trigger Agent 생성 완료!")
print(f"handsoffs: {[agent.name for agent in triage_agent.handoffs]}")

Trigger Agent 생성 완료!
handsoffs: ['rag_specialist', 'summarizer', 'general_assistant']


In [43]:
async def ask_triage_agent(question: str):
    """트리아지 에이전트에게 질문하기"""
    with trace("Multi-Agent RAG"):
        result = await Runner.run(triage_agent, question)
        return result

In [44]:
# 테스트 1: RAG
test1 = "청동기 시대 사람들의 대표적인 무덤이 뭐야?"

print(f"질문: {test1}")
print("=" * 50)

result1 = await ask_triage_agent(test1)
print(f"답변: {result1.final_output}")
print(f"최종 에이전트: {result1.last_agent.name}")

질문: 청동기 시대 사람들의 대표적인 무덤이 뭐야?
답변: 청동기 시대의 대표적인 무덤은 고인돌입니다.
최종 에이전트: rag_specialist


In [45]:
# 테스트 2: 일반 대화
test2 = "안녕!"

print(f"질문: {test2}")
print("=" * 50)

result2 = await ask_triage_agent(test2)
print(f"답변: {result2.final_output}")
print(f"최종 에이전트: {result2.last_agent.name}")

질문: 안녕!
답변: 안녕! 반가워요 😊
최종 에이전트: general_assistant


In [46]:
# 테스트 3: 요약 전문가
test3 = "삼국 시대의 회의제에 대해서 요약해줘."

print(f"질문: {test3}")
print("=" * 50)

result3 = await ask_triage_agent(test3)
print(f"답변: {result3.final_output}")
print(f"최종 에이전트: {result3.last_agent.name}")

질문: 삼국 시대의 회의제에 대해서 요약해줘.
답변: 삼국 시대의 회의제는 왕권을 보좌하고 중요한 국정을 논의하던 귀족 중심의 합의 기구였다.

- 공통점
  - 왕이 독단적으로 통치하지 않고, 유력 귀족들과 함께 국가 중대사를 결정함
  - 왕권과 귀족 세력의 균형을 보여줌
  - 전쟁, 외교, 법률, 인사 같은 중요한 문제를 논의함

- 고구려
  - 제가회의
  - 여러 귀족이 모여 국가의 중대사를 협의
  - 귀족 연합적 성격이 강함

- 백제
  - 정사암 회의
  - 귀족들이 국가 운영을 논의
  - 귀족 세력의 영향이 컸음

- 신라
  - 화백회의
  - 진골 귀족이 참여
  - 합의제적 성격이 강했고, 만장일치 원칙이 특징
  - 국가의 중요한 일을 결정함

- 의의
  - 삼국의 정치가 귀족 합의와 왕권 중심이 함께 작동했음을 보여줌
  - 이후 중앙 집권화가 진행되면서 점차 약화됨

원하면 제가 이걸 표로도 정리해드릴게요.
최종 에이전트: summarizer
